In [2]:
import numpy as np
import os
import sys
import joblib

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    print("ERROR: pip install matplotlib seaborn"); sys.exit(1)

try:
    import tensorflow as tf
    TF_VERSION = tuple(int(x) for x in tf.__version__.split(".")[:2])
    MODEL_EXT  = ".keras" if TF_VERSION >= (2, 16) else ".h5"
except ImportError:
    print("ERROR: pip install tensorflow"); sys.exit(1)

from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report, roc_auc_score


print("="*60)
print("Generating Visualizations...")
print("="*60)

DATA_DIR  = "data"
MODEL_DIR = "models"
PLOTS_DIR = "plots"
TIMESTEPS = joblib.load(os.path.join(DATA_DIR,"lstm_timesteps.save")) \
            if os.path.exists(os.path.join(DATA_DIR,"lstm_timesteps.save")) else 10
os.makedirs(PLOTS_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor":"#1e1e2e","axes.facecolor":"#2a2a3e",
    "axes.edgecolor":"#6e6e8e","axes.labelcolor":"#cdd6f4",
    "axes.titlecolor":"#cdd6f4","xtick.color":"#cdd6f4",
    "ytick.color":"#cdd6f4","text.color":"#cdd6f4",
    "grid.color":"#3e3e5e","grid.alpha":0.4,"font.size":11,
})
C = {"benign":"#89b4fa","attack":"#f38ba8","ae":"#a6e3a1",
     "lstm":"#fab387","ens":"#cba6f7","line":"#f9e2af"}

X_test = np.load(os.path.join(DATA_DIR,"X_test.npy")).astype(np.float32)
y_test = np.load(os.path.join(DATA_DIR,"y_test.npy")).astype(np.int32)

predictions  = {}
model_colors = {"Supervised Autoencoder":C["ae"],"BiLSTM Attention":C["lstm"],"Ensemble":C["ens"]}

# Load AE
ae_probs = None
for ext in [".keras",".h5"]:
    p = os.path.join(MODEL_DIR,f"autoencoder_model{ext}")
    if os.path.exists(p):
        ae = tf.keras.models.load_model(p, compile=False)
        out = ae.predict(X_test, batch_size=1024, verbose=0)
        if isinstance(out,dict): ae_probs = out["classification"].flatten()
        elif isinstance(out,(list,tuple)): ae_probs = next(o.flatten() for o in out if o.shape[-1]==1)
        else: ae_probs = out.flatten()
        predictions["Supervised Autoencoder"] = (y_test, (ae_probs>0.5).astype(int), ae_probs)
        print("Autoencoder loaded.")
        break

# Load LSTM
lstm_probs = None
y_seq      = None
for ext in [".keras",".h5"]:
    p = os.path.join(MODEL_DIR,f"bilstm_model{ext}")
    if os.path.exists(p):
        lstm  = tf.keras.models.load_model(p, compile=False)
        n_seq = len(X_test)-TIMESTEPS
        shape = (n_seq,TIMESTEPS,X_test.shape[1])
        strides=(X_test.strides[0],X_test.strides[0],X_test.strides[1])
        X_seq = np.lib.stride_tricks.as_strided(X_test,shape=shape,strides=strides).copy().astype(np.float32)
        y_seq = np.array([int(np.any(y_test[i:i+TIMESTEPS]==1)) for i in range(n_seq)],dtype=np.int32)
        lstm_probs = lstm.predict(X_seq,batch_size=1024,verbose=0).flatten()
        predictions["BiLSTM Attention"] = (y_seq,(lstm_probs>0.5).astype(int),lstm_probs)
        print("BiLSTM loaded.")
        break

if ae_probs is not None and lstm_probs is not None:
    ep = 0.45*ae_probs[TIMESTEPS:]+0.55*lstm_probs
    predictions["Ensemble"] = (y_seq,(ep>0.5).astype(int),ep)

# ── PLOT 1 — Confusion Matrices ───────────────────────────────────────────────
print("\nPlot 1: Confusion Matrices...")
n = len(predictions)
fig,axes = plt.subplots(1,n,figsize=(6*n,5))
if n==1: axes=[axes]
for ax,(name,(yt,yp,_)) in zip(axes,predictions.items()):
    cm = confusion_matrix(yt,yp)
    cp = cm.astype(float)/cm.sum(axis=1,keepdims=True)*100
    sns.heatmap(cp,annot=False,cmap="Blues",
                xticklabels=["BENIGN","ATTACK"],yticklabels=["BENIGN","ATTACK"],
                ax=ax,cbar=True,linewidths=0.5,linecolor="#3e3e5e")
    for i in range(2):
        for j in range(2):
            col="white" if cp[i,j]>50 else "#cdd6f4"
            ax.text(j+0.5,i+0.38,f"{cm[i,j]:,}",ha="center",va="center",
                    fontsize=13,fontweight="bold",color=col)
            ax.text(j+0.5,i+0.65,f"({cp[i,j]:.1f}%)",ha="center",va="center",
                    fontsize=9,color=col)
    ax.set_title(f"{name}\nConfusion Matrix",fontsize=11,pad=10)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
fig.suptitle("Confusion Matrices — Behavioural Analysis of Network Traffic",
             fontsize=13,fontweight="bold",y=1.02)
plt.tight_layout()
p=os.path.join(PLOTS_DIR,"1_confusion_matrices.png")
plt.savefig(p,dpi=150,bbox_inches="tight",facecolor=fig.get_facecolor())
plt.close(); print(f"  Saved: {p}")

# ── PLOT 2 — ROC Curves ───────────────────────────────────────────────────────
print("Plot 2: ROC Curves...")
fig,ax=plt.subplots(figsize=(8,6))
ax.plot([0,1],[0,1],"--",color="#6e6e8e",lw=1.5,label="Random Guess (AUC=0.50)")
for name,(yt,_,yp) in predictions.items():
    try:
        fpr,tpr,_=roc_curve(yt,yp)
        ra=auc(fpr,tpr)
        ax.plot(fpr,tpr,lw=2.5,color=model_colors.get(name,"#cdd6f4"),
                label=f"{name}  (AUC={ra:.4f})")
        ax.fill_between(fpr,tpr,alpha=0.05,color=model_colors.get(name,"#cdd6f4"))
    except: pass
ax.set_xlabel("False Positive Rate",fontsize=12)
ax.set_ylabel("True Positive Rate",fontsize=12)
ax.set_title("ROC Curves — All DL Models",fontsize=14,fontweight="bold")
ax.legend(loc="lower right",fontsize=10,facecolor="#2a2a3e",edgecolor="#6e6e8e")
ax.grid(True,alpha=0.3); ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
plt.tight_layout()
p=os.path.join(PLOTS_DIR,"2_roc_curves.png")
plt.savefig(p,dpi=150,bbox_inches="tight",facecolor=fig.get_facecolor())
plt.close(); print(f"  Saved: {p}")

# ── PLOT 3 — Model Comparison ─────────────────────────────────────────────────
print("Plot 3: Model Comparison...")
keys=["BENIGN F1","ATTACK F1","Macro F1","Accuracy"]
mdata={}
for name,(yt,yp,_) in predictions.items():
    r=classification_report(yt,yp,output_dict=True,zero_division=0)
    mdata[name]={"BENIGN F1":r["0"]["f1-score"],"ATTACK F1":r["1"]["f1-score"],
                 "Macro F1":r["macro avg"]["f1-score"],"Accuracy":r["accuracy"]}
x=np.arange(len(keys)); w=0.25; n_m=len(predictions)
offsets=np.linspace(-(n_m-1)*w/2,(n_m-1)*w/2,n_m)
fig,ax=plt.subplots(figsize=(11,6))
for i,(name,mv) in enumerate(mdata.items()):
    vals=[mv[k] for k in keys]
    bars=ax.bar(x+offsets[i],vals,w,label=name,
                color=model_colors.get(name,"#cdd6f4"),
                alpha=0.85,edgecolor="#1e1e2e")
    for bar,val in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.005,
                f"{val:.3f}",ha="center",va="bottom",fontsize=8,color="#cdd6f4")
ax.set_xticks(x); ax.set_xticklabels(keys,fontsize=11)
ax.set_ylim([0,1.12]); ax.set_ylabel("Score",fontsize=12)
ax.set_title("DL Model Comparison — Supervised AE vs BiLSTM vs Ensemble",
             fontsize=13,fontweight="bold")
ax.legend(facecolor="#2a2a3e",edgecolor="#6e6e8e",fontsize=10)
ax.axhline(y=0.99,color=C["line"],lw=1.5,linestyle="--",alpha=0.7,label="0.99 target")
ax.grid(axis="y",alpha=0.3)
plt.tight_layout()
p=os.path.join(PLOTS_DIR,"3_model_comparison.png")
plt.savefig(p,dpi=150,bbox_inches="tight",facecolor=fig.get_facecolor())
plt.close(); print(f"  Saved: {p}")

# ── PLOT 4 — Class Distribution ───────────────────────────────────────────────
print("Plot 4: Class Distribution...")
n_b=int((y_test==0).sum()); n_a=int((y_test==1).sum())
fig,axes=plt.subplots(1,2,figsize=(11,5))
bars=axes[0].bar(["BENIGN","ATTACK"],[n_b,n_a],
                 color=[C["benign"],C["attack"]],edgecolor="#1e1e2e",alpha=0.85,width=0.5)
for bar,val in zip(bars,[n_b,n_a]):
    axes[0].text(bar.get_x()+bar.get_width()/2,bar.get_height()+80,
                 f"{val:,}\n({100*val/(n_b+n_a):.1f}%)",
                 ha="center",fontsize=12,color="#cdd6f4")
axes[0].set_title("Test Set Sample Count",fontsize=12,fontweight="bold")
axes[0].set_ylabel("Samples"); axes[0].grid(axis="y",alpha=0.3)
axes[0].set_ylim([0,max(n_b,n_a)*1.25])
wedges,texts,autotexts=axes[1].pie([n_b,n_a],labels=["BENIGN","ATTACK"],
    colors=[C["benign"],C["attack"]],autopct="%1.1f%%",startangle=90,
    wedgeprops={"edgecolor":"#1e1e2e","linewidth":2},
    textprops={"color":"#cdd6f4","fontsize":12})
for at in autotexts: at.set_color("#1e1e2e"); at.set_fontweight("bold")
axes[1].set_title("Class Balance",fontsize=12,fontweight="bold")
fig.suptitle("Dataset Class Distribution (Test Set)",fontsize=13,fontweight="bold")
plt.tight_layout()
p=os.path.join(PLOTS_DIR,"4_class_distribution.png")
plt.savefig(p,dpi=150,bbox_inches="tight",facecolor=fig.get_facecolor())
plt.close(); print(f"  Saved: {p}")

# ── PLOT 5 — Probability Distributions ───────────────────────────────────────
print("Plot 5: Probability Distributions...")
n_plots = sum(1 for name in ["Supervised Autoencoder","BiLSTM Attention"] if name in predictions)
if n_plots > 0:
    fig,axes=plt.subplots(1,n_plots,figsize=(9*n_plots,5))
    if n_plots==1: axes=[axes]
    ax_idx=0
    for name in ["Supervised Autoencoder","BiLSTM Attention"]:
        if name not in predictions: continue
        yt,_,yp=predictions[name]
        axes[ax_idx].hist(yp[yt==0],bins=60,alpha=0.7,color=C["benign"],
                          label="BENIGN",density=True,edgecolor="none")
        axes[ax_idx].hist(yp[yt==1],bins=60,alpha=0.7,color=C["attack"],
                          label="ATTACK",density=True,edgecolor="none")
        axes[ax_idx].axvline(x=0.5,color=C["line"],lw=2,linestyle="--",
                             label="Threshold (0.5)")
        axes[ax_idx].set_xlabel("P(ATTACK)",fontsize=11)
        axes[ax_idx].set_ylabel("Density",fontsize=11)
        axes[ax_idx].set_title(f"{name}\nPrediction Confidence",fontsize=11,fontweight="bold")
        axes[ax_idx].legend(facecolor="#2a2a3e",edgecolor="#6e6e8e",fontsize=9)
        axes[ax_idx].grid(alpha=0.3)
        axes[ax_idx].text(0.02,0.95,"Well-separated = confident model",
                          transform=axes[ax_idx].transAxes,fontsize=8,
                          color="#6e6e8e",style="italic",va="top")
        ax_idx+=1
    plt.suptitle("Prediction Probability Distributions",fontsize=13,fontweight="bold")
    plt.tight_layout()
    p=os.path.join(PLOTS_DIR,"5_probability_distributions.png")
    plt.savefig(p,dpi=150,bbox_inches="tight",facecolor=fig.get_facecolor())
    plt.close(); print(f"  Saved: {p}")

# ── PLOT 6 — Architecture Diagram ────────────────────────────────────────────
print("Plot 6: Architecture Diagram...")
fig,axes=plt.subplots(1,2,figsize=(18,8))
for ax in axes: ax.set_xlim(0,10); ax.set_ylim(0,9); ax.axis("off")

def box(ax,x,y,w,h,label,sub="",color="#89b4fa"):
    r=plt.Rectangle((x,y),w,h,facecolor=color,edgecolor="#cdd6f4",
                     linewidth=1.5,alpha=0.9,zorder=2)
    ax.add_patch(r)
    ax.text(x+w/2,y+h/2+(0.2 if sub else 0),label,
            ha="center",va="center",fontsize=8.5,
            fontweight="bold",color="#1e1e2e",zorder=3)
    if sub: ax.text(x+w/2,y+h/2-0.3,sub,ha="center",va="center",
                    fontsize=7,color="#1e1e2e",zorder=3)

def arr(ax,x1,y1,x2,y2):
    ax.annotate("",xy=(x2,y2),xytext=(x1,y1),
                arrowprops=dict(arrowstyle="->",color="#cdd6f4",lw=1.5),zorder=1)

# AE diagram
ax=axes[0]
ax.text(5,8.4,"Supervised Autoencoder",ha="center",fontsize=13,
        fontweight="bold",color="#cdd6f4")
box(ax,0.3,3.5,1.8,1.4,"INPUT","N features","#89b4fa")
box(ax,2.5,3.5,1.8,1.4,"ENCODER","256→128→64\nBN+Dropout","#a6e3a1")
box(ax,4.8,3.5,1.8,1.4,"BOTTLENECK","32 units","#f9e2af")
box(ax,7.0,5.2,2.0,1.0,"DECODER","→recon","#89dceb")
box(ax,7.0,1.8,2.0,1.0,"CLASSIFIER","→sigmoid","#f38ba8")
box(ax,4.8,6.5,1.8,1.0,"SKIP e3","concat","#6c7086")
arr(ax,2.1,4.2,2.5,4.2); arr(ax,4.3,4.2,4.8,4.2)
arr(ax,6.6,4.6,7.0,5.7); arr(ax,6.6,3.8,7.0,2.3)
arr(ax,5.7,4.9,5.7,6.5); arr(ax,6.6,6.8,7.0,2.5)
ax.text(5,0.5,"Loss = 0.1×MSE + 0.9×FocalLoss",ha="center",
        fontsize=9,color="#a6adc8",style="italic")

# LSTM diagram
ax=axes[1]
ax.text(5,8.4,"BiLSTM with Attention",ha="center",fontsize=13,
        fontweight="bold",color="#cdd6f4")
box(ax,0.3,3.5,1.8,1.4,"INPUT","10 flow\nsequences","#89b4fa")
box(ax,2.5,3.5,1.8,1.4,"BiLSTM\nLayer 1","128 units\n←→","#fab387")
box(ax,4.7,3.5,1.8,1.4,"BiLSTM\nLayer 2","64 units\n←→","#fab387")
box(ax,6.9,3.5,1.8,1.4,"ATTENTION","learns which\nflows matter","#f9e2af")
box(ax,4.2,1.0,3.0,1.2,"CLASSIFIER","Dense(64)→Dense(1)\nsigmoid","#f38ba8")
arr(ax,2.1,4.2,2.5,4.2); arr(ax,4.3,4.2,4.7,4.2)
arr(ax,6.5,4.2,6.9,4.2); arr(ax,7.8,3.5,5.7,2.2)
ax.text(5,0.3,"Early Detection: flags attack before sequence completes",
        ha="center",fontsize=9,color="#a6adc8",style="italic")

plt.suptitle("Deep Learning Model Architectures — Behavioural Analysis of Network Traffic",
             fontsize=13,fontweight="bold",y=0.98)
plt.tight_layout()
p=os.path.join(PLOTS_DIR,"6_architecture_diagrams.png")
plt.savefig(p,dpi=150,bbox_inches="tight",facecolor=fig.get_facecolor())
plt.close(); print(f"  Saved: {p}")

print(f"\nAll plots saved to '{PLOTS_DIR}/':")
for f in sorted(os.listdir(PLOTS_DIR)):
    if f.endswith(".png"): print(f"  {f}")
print("visualize.py completed successfully!")

Generating Visualizations...
Autoencoder loaded.
BiLSTM loaded.

Plot 1: Confusion Matrices...
  Saved: plots\1_confusion_matrices.png
Plot 2: ROC Curves...
  Saved: plots\2_roc_curves.png
Plot 3: Model Comparison...
  Saved: plots\3_model_comparison.png
Plot 4: Class Distribution...
  Saved: plots\4_class_distribution.png
Plot 5: Probability Distributions...
  Saved: plots\5_probability_distributions.png
Plot 6: Architecture Diagram...
  Saved: plots\6_architecture_diagrams.png

All plots saved to 'plots/':
  1_confusion_matrices.png
  2_roc_curves.png
  3_model_comparison.png
  4_class_distribution.png
  5_probability_distributions.png
  6_architecture_diagrams.png
visualize.py completed successfully!
